# 02 — Dimensionality Reduction (PCA) on Gene Indicator Features

Adapted from [SalvatoreRa/tutorial — complexity reduction techniques](https://github.com/SalvatoreRa/tutorial/blob/main/genomic%20series/complexity%20reduction%20techniques.ipynb) (Apache-2.0).

**Purpose:** Apply PCA and variance-explained analysis to the `G__*` gene indicator columns from the merged GENIE dataset. Identify dominant axes of genomic variation and visualize sample clustering in reduced space.

**Inputs:**
- `datasets_analysis_dictionary/merged_genie.xlsx` or a processed CSV in `data/processed/`

**Outputs:**
- Scree plot, biplot, cumulative variance plot → `reports/figures/`
- PCA-transformed features → `data/features/`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline

# ── Paths ────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH    = os.path.join(PROJECT_ROOT, "datasets_analysis_dictionary", "merged_genie.xlsx")
FIG_DIR      = os.path.join(PROJECT_ROOT, "reports", "figures")
FEAT_DIR     = os.path.join(PROJECT_ROOT, "data", "features")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

In [ ]:
# ── Load data ────────────────────────────────────────────────────────
df = pd.read_excel(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Select gene indicator columns (G__*) ────────────────────────────
gene_cols = [c for c in df.columns if c.startswith("G__")]
print(f"Gene indicator columns found: {len(gene_cols)}")

X = df[gene_cols].dropna()
print(f"Samples after dropping NaN: {X.shape[0]}")

In [ ]:
# ── Standardize and fit PCA ──────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)
print(f"Components needed for 90% variance: {np.argmax(cumulative >= 0.90) + 1}")

In [ ]:
# ── Scree plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_show = min(30, len(explained))
axes[0].bar(range(1, n_show + 1), explained[:n_show], color="steelblue")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Variance Explained")
axes[0].set_title("Scree Plot")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="darkorange")
axes[1].axhline(0.90, ls="--", color="grey", label="90% threshold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance Explained")
axes[1].set_title("Cumulative Variance")
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pca_scree_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 2-D scatter (PC1 vs PC2) ────────────────────────────────────────
# Color by a stratification variable if available (e.g. SUBTYPE)
plot_df = pd.DataFrame({"PC1": X_pca[:, 0], "PC2": X_pca[:, 1]})

strat_col = None
for candidate in ["SUBTYPE", "MANTIS_BIN", "CANCER_TYPE"]:
    if candidate in df.columns:
        strat_col = candidate
        plot_df[strat_col] = df.loc[X.index, strat_col].values
        break

fig, ax = plt.subplots(figsize=(9, 7))
if strat_col:
    sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue=strat_col, alpha=0.6, ax=ax)
else:
    ax.scatter(plot_df["PC1"], plot_df["PC2"], alpha=0.5)

ax.set_title("PCA — Gene Indicator Features")
ax.set_xlabel(f"PC1 ({explained[0]:.1%})")
ax.set_ylabel(f"PC2 ({explained[1]:.1%})")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "pca_scatter_pc1_pc2.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Export PCA-transformed features ──────────────────────────────────
n_keep = np.argmax(cumulative >= 0.90) + 1
pca_df = pd.DataFrame(
    X_pca[:, :n_keep],
    columns=[f"PC{i+1}" for i in range(n_keep)],
    index=X.index,
)
pca_df.to_csv(os.path.join(FEAT_DIR, "pca_gene_features.csv"))
print(f"Saved {n_keep} PCA components → data/features/pca_gene_features.csv")